In [1]:
%matplotlib inline 
import json
import numpy as np
import operator
import os
import pandas
import pylab as plt
import random
from ase.build import bulk
from ase.data import reference_states, atomic_numbers
from sklearn.neighbors import KernelDensity
# from pyiron_atomistics import Project

In [2]:
import matplotlib as mpl
mpl.rc('font', family='Times New Roman')
plt.rcParams["mathtext.fontset"] = "stix"

In [3]:
from pyiron_atomistics.thermodynamics.interfacemethod import freeze_one_half, remove_selective_dynamics, set_server, create_job_template, fix_iso, fix_z_dir, half_velocity, minimize_pos, minimize_vol, next_calc, npt_solid, npt_liquid, next_step_funct, round_temperature_next, strain_circle, analyse_minimized_structure, get_press, get_center_point, get_strain_lst, get_nve_job_name, plot_solid_liquid_ratio, ratio_selection, plot_equilibration, plot_melting_point_prediction, calc_temp_iteration, get_initial_melting_temperature_guess, validate_convergence, initialise_iterators, get_voronoi_volume, check_for_holes

/Users/janssen/mambaforge/lib/python3.12/site-packages/nglview/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [4]:
input_file = os.path.join(".", 'input.json')
output_file = os.path.join(".", 'output.json')

In [5]:
# Parameters
input_file = "input.json"
output_file = "output.json"


In [6]:
if os.path.exists(input_file) and os.stat(input_file).st_size != 0:
    with open(input_file, 'r') as f:
        input_dict = json.load(f)
else:
    input_dict = {
        "config": [
            "pair_style morse 9.97749\n",
            "pair_coeff * * 0.4174 1.3885 2.845\n"
        ],
        "species": ["Al"],
        "element": "Al"
    }


In [7]:
input_dict

{'config': ['pair_style eam/fs \n', 'pair_coeff * * Al1.eam.fs Al\n'],
 'filename': './Al1.eam.fs',
 'species': ['Al'],
 'element': 'Al'}

In [8]:
pot_dict = input_dict.copy()

In [9]:
if 'model' not in pot_dict.keys():
    pot_dict['model'] = 'Lammps'
if 'name' not in pot_dict.keys():
    pot_dict['name'] = 'CustomPotential'
if 'filename' not in pot_dict.keys():
    pot_path = []
else:
    pot_path = [os.path.abspath(pot_dict['filename'])]
potential = pandas.DataFrame({'Config': [pot_dict['config']],
                             'Filename': [pot_path],
                             'Model': [pot_dict['model']],
                             'Name': [pot_dict['name']],
                             'Species': [pot_dict['species']]
                             })

In [10]:
project_parameter = {
    # 'project': pr,
    'run_time_steps': 50000,
    'nvt_run_time_steps': 10000,
    'nve_run_time_steps': 10000,
    'temperature_left': 0,
    'temperature_right': 1000,
    'strain_run_time_steps': 1000,
    'convergence_criterion': 1,
    'potential': potential,
    'cpu_cores': 1, 
    # 'job_type': pr.job_type.Lammps,
    'enable_h5md': False,
    'points': 21,
    'boundary_value': 0.25,
    'ratio_boundary': 0.25,
    'timestep_lst': [2, 2, 1],
    'fit_range_lst': [0.05, 0.01, 0.01],
    'nve_run_time_steps_lst': [25000, 20000, 50000],
    'number_of_atoms': 8000, 
}

In [11]:
for k in input_dict.keys():
    project_parameter[k] = input_dict[k]

In [12]:
if 'crystalstructure' not in project_parameter.keys():
    project_parameter['crystalstructure'] = reference_states[atomic_numbers[project_parameter['element']]]['symmetry']

In [13]:
if 'seed' not in project_parameter.keys():
    project_parameter['seed'] = random.randint(0,99999)
project_parameter['seed']

53976

In [14]:
# Values from a previous calculation can be inserted here to reproduce the results 
step_dict = {}
if os.path.exists(output_file):
    with open(output_file, 'r') as f:
        step_dict_str = json.load(f)
    for k,v in step_dict_str.items():
        step_dict[int(k)] = v
step_dict

{}

# From here on the notebook is automated - no change required !

In [15]:
step_count = 0
temperature_next = None
enable_iteration = True
convergence_goal_achieved = False

In [16]:
if 'lattice_constant' in project_parameter.keys():
    a = project_parameter['lattice_constant']
else:
    a = None
if project_parameter['crystalstructure'] == 'hcp':
    basis = bulk(name=project_parameter['element'], crystalstructure=project_parameter['crystalstructure'].lower(), a=a, orthorhombic=True)
else:
    basis = bulk(name=project_parameter['element'], crystalstructure=project_parameter['crystalstructure'].lower(), a=a, cubic=True)
basis_lst = [basis.repeat([i, i, i]) for i in range(5,30)]
basis = basis_lst[np.argmin([np.abs(len(b)-project_parameter['number_of_atoms']/2) for b in basis_lst])]

In [17]:
from structuretoolkit.analyse import get_adaptive_cna_descriptors
from structuretoolkit.visualize import plot3d

In [18]:
get_adaptive_cna_descriptors(structure=basis)

{'others': 0, 'fcc': 4000, 'hcp': 0, 'bcc': 0, 'ico': 0}

In [19]:
plot3d(basis)

NGLWidget()

In [20]:
timestep_iter, fit_range_iter, nve_run_time_steps_iter = initialise_iterators(project_parameter)
timestep = next(timestep_iter)
fit_range = next(fit_range_iter)
nve_run_time_steps = next(nve_run_time_steps_iter)

# Step 1: set up the solid sample and roughly estimate a melting point

In [21]:
from atomistics.calculators.lammps import optimize_positions_and_volume_with_lammpslib, optimize_positions_with_lammpslib, calc_molecular_dynamics_npt_with_lammpslib
from atomistics.calculators.lammps.commands import LAMMPS_RUN, LAMMPS_MINIMIZE_VOLUME, LAMMPS_THERMO_STYLE, LAMMPS_THERMO, LAMMPS_MINIMIZE, LAMMPS_TIMESTEP, LAMMPS_VELOCITY, LAMMPS_ENSEMBLE_NPT
from atomistics.calculators.lammps.libcalculator import lammps_run, lammps_shutdown, lammps_calc_md

/Users/janssen/projects/atomistics/atomistics/calculators/__init__.py:63: UserWarning: calc_static_with_qe(), evaluate_with_qe() and optimize_positions_and_volume_with_qe() are not available as the import of the module named 'pwtools' failed.
  raise_warning(module_list=quantum_espresso_function, import_error=e)


In [22]:
from ase.atoms import Atoms 
from pandas import DataFrame

In [23]:
from jinja2 import Template

In [24]:
LAMMPS_MINIMIZE_VOLUME

'fix ensemble all box/relax iso 0.0'

In [25]:
def optimize_positions_and_volume_with_lammpslib(
    structure: Atoms,
    potential_dataframe: DataFrame,
    min_style: str = "cg",
    etol: float = 0.0,
    ftol: float = 0.0001,
    maxiter: int = 100000,
    maxeval: int = 10000000,
    thermo: int = 10,
    lmp=None,
    **kwargs,
) -> Atoms:
    template_str = (
        LAMMPS_MINIMIZE_VOLUME + " vmax 0.001"
        + "\n"
        + LAMMPS_THERMO_STYLE
        + "\n"
        + LAMMPS_THERMO
        + "\n"
        + LAMMPS_MINIMIZE
    )
    lmp_instance = lammps_run(
        structure=structure,
        potential_dataframe=potential_dataframe,
        input_template=Template(template_str).render(
            min_style=min_style,
            etol=etol,
            ftol=ftol,
            maxiter=maxiter,
            maxeval=maxeval,
            thermo=thermo,
        ),
        lmp=lmp,
        **kwargs,
    )
    structure_copy = structure.copy()
    structure_copy.set_cell(lmp_instance.interactive_cells_getter(), scale_atoms=True)
    structure_copy.positions = lmp_instance.interactive_positions_getter()
    lammps_shutdown(lmp_instance=lmp_instance, close_instance=lmp is None)
    return structure_copy

In [26]:
potential

,Config,Filename,Model,Name,Species
0,"[pair_style eam/fs \n, pair_coeff * * Al1.eam....",[/Users/janssen/notebooks/2026/2026-01-08-melt...,Lammps,CustomPotential,[Al]


In [27]:
structure_opt = optimize_positions_and_volume_with_lammpslib(
    structure=basis,
    potential_dataframe=potential,
    min_style="cg",
    etol=0.0,
    ftol=0.0001,
    maxiter=100000,
    maxeval=10000000,
    thermo=10,
    lmp=None,
)

In [28]:
structure_opt

Atoms(symbols='Al4000', pbc=True, cell=[40.452597982401535, 40.452597982401535, 40.452597982401535])

In [29]:
from pyiron_atomistics.thermodynamics.interfacemethod import analyse_minimized_structure, next_calc, next_step_funct, check_diamond, analyse_structure

In [30]:
from structuretoolkit.analyse import get_adaptive_cna_descriptors, get_diamond_structure_descriptors

In [31]:
def check_diamond(structure):
    """
    Utility function to check if the structure is fcc, bcc, hcp or diamond

    Args:
        structure (pyiron_atomistics.structure.atoms.Atoms): Atomistic Structure object to check

    Returns:
        bool: true if diamond else false
    """
    cna_dict = get_adaptive_cna_descriptors(
        structure=structure, mode="total", ovito_compatibility=True
    )
    dia_dict = get_diamond_structure_descriptors(
        structure=structure, mode="total", ovito_compatibility=True
    )
    return (
        cna_dict["CommonNeighborAnalysis.counts.OTHER"]
        > dia_dict["IdentifyDiamond.counts.OTHER"]
    )

In [32]:
def analyse_structure(structure, mode="total", diamond=False):
    """
    Use either common neighbor analysis or the diamond structure detector

    Args:
        structure (pyiron_atomistics.structure.atoms.Atoms): The structure to analyze.
        mode ("total"/"numeric"/"str"): Controls the style and level
            of detail of the output.
            - total : return number of atoms belonging to each structure
            - numeric : return a per atom list of numbers- 0 for unknown,
                1 fcc, 2 hcp, 3 bcc and 4 icosa
            - str : return a per atom string of sructures
        diamond (bool): Flag to either use the diamond structure detector or
            the common neighbor analysis.

    Returns:
        (depends on `mode`)
    """
    if not diamond:
        return get_adaptive_cna_descriptors(
            structure=structure, mode=mode, ovito_compatibility=True
        )
    else:
        return get_diamond_structure_descriptors(
            structure=structure, mode=mode, ovito_compatibility=True
        )

In [33]:
def analyse_minimized_structure(structure):
    """

    Args:
        ham (GenericJob):

    Returns:

    """
    diamond_flag = check_diamond(structure=structure)
    final_structure_dict = analyse_structure(
        structure=structure, mode="total", diamond=diamond_flag
    )
    key_max = max(final_structure_dict.items(), key=operator.itemgetter(1))[0]
    number_of_atoms = len(structure)
    distribution_initial = final_structure_dict[key_max] / number_of_atoms
    distribution_initial_half = distribution_initial / 2
    return (
        structure,
        key_max,
        number_of_atoms,
        distribution_initial_half,
        final_structure_dict,
    )

In [34]:
(
    structure_after_minimization,
    key_max,
    number_of_atoms,
    distribution_initial_half,
    _,
) = analyse_minimized_structure(structure=structure_opt)

In [35]:
key_max, number_of_atoms, distribution_initial_half,

('CommonNeighborAnalysis.counts.FCC', 4000, 0.5)

In [36]:
temperature_left = project_parameter["temperature_left"]
temperature_right = project_parameter["temperature_right"]
temperature_left, temperature_right

(0, 1000)

In [37]:
from atomistics.shared.output import OutputMolecularDynamics

In [38]:
LAMMPS_VELOCITY = 'velocity all create {{ temp }} {{seed}} dist {{dist}}'

In [39]:
def calc_molecular_dynamics_npt_with_lammpslib(
    structure: Atoms,
    potential_dataframe: pandas.DataFrame,
    Tstart: float = 100.0,
    Tstop: float = 100.0,
    Tdamp: float = 0.1,
    run: int = 100,
    thermo: int = 100,
    timestep: float = 0.001,
    Pstart: float = 0.0,
    Pstop: float = 0.0,
    Pdamp: float = 1.0,
    seed: int = 4928459,
    dist: str = "gaussian",
    disable_initial_velocity: bool = False,
    lmp=None,
    output_keys=OutputMolecularDynamics.keys(),
    **kwargs,
) -> dict:
    if not disable_initial_velocity:
        init_str = (
            LAMMPS_THERMO_STYLE
            + "\n"
            + LAMMPS_TIMESTEP
            + "\n"
            + LAMMPS_THERMO
            + "\n"
            + LAMMPS_VELOCITY
            + "\n"
            + LAMMPS_ENSEMBLE_NPT + " couple xyz"
        )
        input_template = Template(init_str).render(
            thermo=thermo,
            Tstart=Tstart,
            temp=Tstart,
            Tstop=Tstop,
            Tdamp=Tdamp,
            Pstart=Pstart,
            Pstop=Pstop,
            Pdamp=Pdamp,
            timestep=timestep,
            seed=seed,
            dist=dist,
        )
    else:
        init_str = (
            LAMMPS_THERMO_STYLE
            + "\n"
            + LAMMPS_TIMESTEP
            + "\n"
            + LAMMPS_THERMO
            + "\n"
            + LAMMPS_ENSEMBLE_NPT + " couple xyz"
        )
        input_template = Template(init_str).render(
            thermo=thermo,
            Tstart=Tstart,
            temp=Tstart,
            Tstop=Tstop,
            Tdamp=Tdamp,
            Pstart=Pstart,
            Pstop=Pstop,
            Pdamp=Pdamp,
            timestep=timestep,
        )
    run_str = LAMMPS_RUN + "\n"
    print(input_template)
    lmp_instance = lammps_run(
        structure=structure,
        potential_dataframe=potential_dataframe,
        input_template=input_template,
        lmp=lmp,
        **kwargs,
    )
    result_dict = lammps_calc_md(
        lmp_instance=lmp_instance,
        run_str=run_str,
        run=run,
        thermo=thermo,
        output_keys=output_keys,
    )
    lammps_shutdown(lmp_instance=lmp_instance, close_instance=lmp is None)
    return result_dict

In [40]:
def next_calc(structure, temperature, project_parameter, run_time_steps=10000):
    """
    Calculate NPT ensemble at a given temperature using the job defined in the project parameters:
    - job_type: Type of Simulation code to be used
    - project: Project object used to create the job
    - potential: Interatomic Potential
    - queue (optional): HPC Job queue to be used

    Args:
        structure (pyiron_atomistics.structure.atoms.Atoms): Atomistic Structure object to be set to the job as input sturcture
        temperature (float): Temperature of the Molecular dynamics calculation
        project_parameter (dict): Dictionary with the project parameters
        run_time_steps (int): Number of Molecular dynamics steps

    Returns:
        Final Atomistic Structure object
    """
    output_md_dict = calc_molecular_dynamics_npt_with_lammpslib(
        structure=structure_opt,
        potential_dataframe=potential,
        Tstart=temperature,
        Tstop=temperature,
        Tdamp=0.1,
        run=run_time_steps,
        thermo=1000,
        timestep=0.001,
        Pstart=0.0,
        Pstop=0.0,
        Pdamp=1.0,
        seed=project_parameter["seed"],
        dist="gaussian",
        disable_initial_velocity=False,
        lmp=None,
        output_keys=OutputMolecularDynamics.keys(),
    )
    structure_md = structure_opt.copy()
    structure_md.set_positions(output_md_dict['positions'][-1])
    structure_md.set_cell(output_md_dict['cell'][-1])
    return structure_md

In [41]:
structure_left = structure_after_minimization
structure_right = next_calc(
    structure=structure_after_minimization,
    temperature=temperature_right,
    project_parameter=project_parameter,
    run_time_steps=project_parameter["strain_run_time_steps"],
)
temperature_step = temperature_right - temperature_left
temperature_step

thermo_style custom step temp pe etotal pxx pxy pxz pyy pyz pzz vol
thermo_modify format float %20.15g
timestep 0.001
thermo 1000
velocity all create 1000 53976 dist gaussian
fix ensemble all npt temp 1000 1000 0.1 iso 0.0 0.0 1.0 couple xyz


1000

In [42]:
def next_step_funct(
    number_of_atoms,
    key_max,
    structure_left,
    structure_right,
    temperature_left,
    temperature_right,
    distribution_initial_half,
    structure_after_minimization,
    run_time_steps,
    project_parameter,
):
    """

    Args:
        number_of_atoms:
        key_max:
        structure_left:
        structure_right:
        temperature_left:
        temperature_right:
        distribution_initial_half:
        structure_after_minimization:
        run_time_steps:
        project_parameter:

    Returns:

    """
    structure_left_dict = analyse_structure(
        structure=structure_left,
        mode="total",
        diamond=project_parameter["crystalstructure"].lower() == "diamond",
    )
    structure_right_dict = analyse_structure(
        structure=structure_right,
        mode="total",
        diamond=project_parameter["crystalstructure"].lower() == "diamond",
    )
    temperature_diff = temperature_right - temperature_left
    if (
        structure_left_dict[key_max] / number_of_atoms > distribution_initial_half
        and structure_right_dict[key_max] / number_of_atoms > distribution_initial_half
    ):
        structure_left = structure_right.copy()
        temperature_left = temperature_right
        temperature_right += temperature_diff
        structure_right = next_calc(
            structure=structure_after_minimization,
            temperature=temperature_right,
            project_parameter=project_parameter,
            run_time_steps=run_time_steps,
        )
    elif (
        structure_left_dict[key_max] / number_of_atoms
        > distribution_initial_half
        > structure_right_dict[key_max] / number_of_atoms
    ):
        temperature_diff /= 2
        temperature_left += temperature_diff
        structure_left = next_calc(
            structure=structure_after_minimization,
            temperature=temperature_left,
            project_parameter=project_parameter,
            run_time_steps=run_time_steps,
        )
    elif (
        structure_left_dict[key_max] / number_of_atoms < distribution_initial_half
        and structure_right_dict[key_max] / number_of_atoms < distribution_initial_half
    ):
        temperature_diff /= 2
        temperature_right = temperature_left
        temperature_left -= temperature_diff
        structure_right = structure_left.copy()
        structure_left = next_calc(
            structure=structure_after_minimization,
            temperature=temperature_left,
            project_parameter=project_parameter,
            run_time_steps=run_time_steps,
        )
    else:
        raise ValueError("We should never reach this point!")
    return structure_left, structure_right, temperature_left, temperature_right

In [43]:
(
    structure_left,
    structure_right,
    temperature_left,
    temperature_right,
) = next_step_funct(
    number_of_atoms=number_of_atoms,
    key_max=key_max,
    structure_left=structure_left,
    structure_right=structure_right,
    temperature_left=temperature_left,
    temperature_right=temperature_right,
    distribution_initial_half=distribution_initial_half,
    structure_after_minimization=structure_after_minimization,
    run_time_steps=project_parameter["strain_run_time_steps"],
    project_parameter=project_parameter,
)

thermo_style custom step temp pe etotal pxx pxy pxz pyy pyz pzz vol
thermo_modify format float %20.15g
timestep 0.001
thermo 1000
velocity all create 500.0 53976 dist gaussian
fix ensemble all npt temp 500.0 500.0 0.1 iso 0.0 0.0 1.0 couple xyz


In [44]:
temperature_step = temperature_right - temperature_left
temperature_step

500.0

In [45]:
while temperature_step > 10:
    (
        structure_left,
        structure_right,
        temperature_left,
        temperature_right,
    ) = next_step_funct(
        number_of_atoms=number_of_atoms,
        key_max=key_max,
        structure_left=structure_left,
        structure_right=structure_right,
        temperature_left=temperature_left,
        temperature_right=temperature_right,
        distribution_initial_half=distribution_initial_half,
        structure_after_minimization=structure_after_minimization,
        run_time_steps=project_parameter["strain_run_time_steps"],
        project_parameter=project_parameter,
    )
    temperature_step = temperature_right - temperature_left
temperature_next = int(round(temperature_left))

thermo_style custom step temp pe etotal pxx pxy pxz pyy pyz pzz vol
thermo_modify format float %20.15g
timestep 0.001
thermo 1000
velocity all create 750.0 53976 dist gaussian
fix ensemble all npt temp 750.0 750.0 0.1 iso 0.0 0.0 1.0 couple xyz
thermo_style custom step temp pe etotal pxx pxy pxz pyy pyz pzz vol
thermo_modify format float %20.15g
timestep 0.001
thermo 1000
velocity all create 875.0 53976 dist gaussian
fix ensemble all npt temp 875.0 875.0 0.1 iso 0.0 0.0 1.0 couple xyz
thermo_style custom step temp pe etotal pxx pxy pxz pyy pyz pzz vol
thermo_modify format float %20.15g
timestep 0.001
thermo 1000
velocity all create 937.5 53976 dist gaussian
fix ensemble all npt temp 937.5 937.5 0.1 iso 0.0 0.0 1.0 couple xyz
thermo_style custom step temp pe etotal pxx pxy pxz pyy pyz pzz vol
thermo_modify format float %20.15g
timestep 0.001
thermo 1000
velocity all create 906.25 53976 dist gaussian
fix ensemble all npt temp 906.25 906.25 0.1 iso 0.0 0.0 1.0 couple xyz
thermo_style cust

In [46]:
temperature_next

914